In [1]:
"""
================================================================================
LSTM vs Transformer for Time Series Analysis
================================================================================

Comparing LSTM vs Transformer for time series analysis is actually a very 
common and useful experiment. People do it for several important reasons in 
machine learning research, finance, operation, production systems, etc.

Some of the more common reasons for doing this kind of analysis include:
1. To determine which architecture models temporal dependencies better
2. To evaluate modeling tradeoffs
3. To test modern architectures against established methods                                                                                                                    
4. To understand the structure of the time series
5. To determine production deployment choices
6. To benchmark performance improvements
7. To measure robustness of models
8. To build explainable AI
9. To push forward time series research
10. To understand anomaly detection behavior

This script performs the following:
1. Downloads historical stock price data (MSFT).
2. Engineers features (returns, volatility, momentum, etc.).
3. Creates weak anomaly labels based on extreme returns.
4. Trains an LSTM Autoencoder and a Transformer Autoencoder for anomaly detection.
5. Compares performance (ROC-AUC, PR-AUC).
6. Trains Transformer Forecasters for 1-day, 5-day, and 21-day horizons.
7. Generates forecasts with prediction intervals.

================================================================================
"""

import math
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, roc_auc_score, average_precision_score
from statistics import NormalDist

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Set matplotlib backend for reliable PNG saving
plt.switch_backend('Agg')

# ==============================================================================
# Configuration
# ==============================================================================

ticker = "MSFT"
SEQ_LEN = 30   # 30 trading days ~ 1.5 months
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = 25
LEARNING_RATE = 1e-3
BATCH_SIZE = 64
TRAIN_SPLIT = 0.7

print(f"Using device: {DEVICE}")

# ==============================================================================
# 1. Data Loading & Feature Engineering
# ==============================================================================

def load_and_prepare_data(ticker):
    """
    Download historical stock price data and engineer features.
    """
    # Download data
    df = yf.Ticker(ticker).history(period="5y", auto_adjust=True)
    df = df[["Open", "High", "Low", "Close", "Volume"]].dropna()
    print(f"Data shape: {df.shape}")
    print(df.tail())

    # --- Feature Set 1: General Features ---
    features = pd.DataFrame(index=df.index)

    # daily return
    features["ret_1d"] = df["Close"].pct_change()

    # rolling volatility
    features["vol_20d"] = features["ret_1d"].rolling(20).std()

    # short and medium momentum
    features["mom_5d"] = df["Close"].pct_change(5)
    features["mom_20d"] = df["Close"].pct_change(20)

    # drawdown from rolling peak
    features["drawdown"] = df["Close"] / df["Close"].cummax() - 1

    # volume shock
    features["vol_chg"] = df["Volume"].pct_change()
    features["vol_z_20"] = ((df["Volume"] - df["Volume"].rolling(20).mean()) / 
                            df["Volume"].rolling(20).std())

    features = features.dropna()

    # --- Feature Set 2: Modeling Features & Anomaly Labels ---
    feat = df.copy()

    feat["log_close"] = np.log(feat["Close"])
    feat["log_return"] = feat["log_close"].diff()

    feat["range_pct"] = (feat["High"] - feat["Low"]) / feat["Close"]
    feat["open_close_pct"] = (feat["Close"] - feat["Open"]) / feat["Open"]

    feat["vol_chg"] = np.log1p(feat["Volume"]).diff()
    feat["ma_10"] = feat["Close"].rolling(10).mean()
    feat["ma_gap_10"] = (feat["Close"] - feat["ma_10"]) / feat["ma_10"]

    feat["realized_vol_10"] = feat["log_return"].rolling(10).std()

    feat = feat.dropna()

    feature_cols = [
        "log_return",
        "range_pct",
        "open_close_pct",
        "vol_chg",
        "ma_gap_10",
        "realized_vol_10",
    ]

    # Build a weak anomaly label for evaluation (Top 1% absolute log returns)
    feat["abs_return"] = feat["log_return"].abs()
    threshold_label = feat["abs_return"].quantile(0.99)
    feat["anomaly_label"] = (feat["abs_return"] >= threshold_label).astype(int)
    
    return df, feat, feature_cols, features

# ==============================================================================
# 2. Data Preprocessing & Windowing
# ==============================================================================

def prepare_sequences(X_raw, y_proxy, feature_cols, seq_len=30, split_idx_ratio=0.7):
    """
    Split data, scale, and create rolling windows for sequence models.
    """
    X_raw = X_raw[feature_cols].copy()
    
    split_idx = int(len(X_raw) * split_idx_ratio)

    X_train_raw = X_raw.iloc[:split_idx]
    X_test_raw  = X_raw.iloc[split_idx:]

    y_train = y_proxy[:split_idx]
    y_test  = y_proxy[split_idx:]
    
    # Keep test indices for later alignment
    test_indices = X_test_raw.index[seq_len:] 

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw)
    X_test  = scaler.transform(X_test_raw)

    # Convert to rolling windows
    def make_windows(X, y, seq_len=30):
        xs, ys = [], []
        for i in range(seq_len, len(X)):
            xs.append(X[i-seq_len:i])
            ys.append(y[i])  # label attached to last day of the window
        return np.array(xs, dtype=np.float32), np.array(ys, dtype=np.int64)

    X_train_seq, y_train_seq = make_windows(X_train, y_train, seq_len)
    X_test_seq, y_test_seq   = make_windows(X_test, y_test, seq_len)

    print(f"Train shape: {X_train_seq.shape}, Test shape: {X_test_seq.shape}")
    
    return X_train_seq, y_train_seq, X_test_seq, y_test_seq, scaler, test_indices

# ==============================================================================
# 3. PyTorch Datasets
# ==============================================================================

class SeqDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = None if y is None else torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is None:
            return self.X[idx]
        return self.X[idx], self.y[idx]

class ForecastDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# ==============================================================================
# 4. Model Definitions
# ==============================================================================

# Model 1: LSTM Autoencoder
class LSTMAutoencoder(nn.Module):
    def __init__(self, n_features, hidden_size=64, latent_size=32, num_layers=1):
        super().__init__()
        self.encoder = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )
        self.to_latent = nn.Linear(hidden_size, latent_size)
        self.from_latent = nn.Linear(latent_size, hidden_size)

        self.decoder = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )
        self.output_layer = nn.Linear(hidden_size, n_features)

    def forward(self, x):
        # x: [B, T, F]
        enc_out, (h_n, c_n) = self.encoder(x)
        h_last = enc_out[:, -1, :]                    # [B, hidden]
        z = self.to_latent(h_last)                    # [B, latent]
        dec_seed = self.from_latent(z).unsqueeze(1)   # [B, 1, hidden]
        dec_input = dec_seed.repeat(1, x.size(1), 1)  # [B, T, hidden]
        dec_out, _ = self.decoder(dec_input)
        out = self.output_layer(dec_out)
        return out
        
# Model 2: Transformer Autoencoder
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)

        pe = pe.unsqueeze(0)  # [1, max_len, d_model]
        self.register_buffer("pe", pe)

    def forward(self, x):
        # x: [B, T, D]
        return x + self.pe[:, :x.size(1), :]

class TransformerAutoencoder(nn.Module):
    def __init__(self, n_features, d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.pos_encoder = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.decoder_mlp = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, n_features)
        )

    def forward(self, x):
        # x: [B, T, F]
        h = self.input_proj(x)
        h = self.pos_encoder(h)
        z = self.encoder(h)         # [B, T, d_model]
        out = self.decoder_mlp(z)   # reconstruct each timestep
        return out

# Model 3: Transformer Forecaster
class TransformerForecaster(nn.Module):
    def __init__(self, n_features, d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.pos_encoder = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1)
        )

    def forward(self, x):
        h = self.input_proj(x)
        h = self.pos_encoder(h)
        z = self.encoder(h)
        last_token = z[:, -1, :]
        out = self.head(last_token)
        return out.squeeze(-1)

# ==============================================================================
# 5. Training Utilities
# ==============================================================================

def train_autoencoder(model, train_loader, epochs=25, lr=1e-3):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    history = []

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        for xb, _ in train_loader:
            xb = xb.to(DEVICE)

            optimizer.zero_grad()
            recon = model(xb)
            loss = criterion(recon, xb)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * xb.size(0)

        epoch_loss = total_loss / len(train_loader.dataset)
        history.append(epoch_loss)
        print(f"Epoch {epoch+1:02d} | loss={epoch_loss:.6f}")

    return history

def reconstruction_scores(model, loader):
    model.eval()
    scores = []
    recons = []
    originals = []

    with torch.no_grad():
        for xb, _ in loader:
            xb = xb.to(DEVICE)
            recon = model(xb)

            # mean squared reconstruction error per window
            err = ((recon - xb) ** 2).mean(dim=(1, 2)).detach().cpu().numpy()
            scores.extend(err)

            recons.append(recon.cpu().numpy())
            originals.append(xb.cpu().numpy())

    return np.array(scores), np.concatenate(originals), np.concatenate(recons)

def train_forecaster(model, train_loader, epochs=25, lr=1e-3):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    history = []

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * xb.size(0)

        epoch_loss = total_loss / len(train_loader.dataset)
        history.append(epoch_loss)
        print(f"Epoch {epoch+1:02d} | loss={epoch_loss:.6f}")

    return history

def make_windows_forecast(X, y, seq_len=30):
    xs, ys = [], []
    for i in range(seq_len, len(X)):
        xs.append(X[i-seq_len:i])
        ys.append(y[i])
    return np.array(xs, dtype=np.float32), np.array(ys, dtype=np.float32)

def train_horizon_model(feat, feature_cols, horizon, seq_len=30, epochs=25, lr=1e-3):
    temp = feat.copy()

    target_col = f"target_{horizon}d"
    temp[target_col] = temp["log_close"].shift(-horizon) - temp["log_close"]

    temp = temp.dropna().copy()

    X_raw = temp[feature_cols].copy()
    y_raw = temp[target_col].values

    split_idx = int(len(X_raw) * 0.7)

    X_train_raw = X_raw.iloc[:split_idx]
    X_test_raw  = X_raw.iloc[split_idx:]

    y_train_raw = y_raw[:split_idx]
    y_test_raw = y_raw[split_idx:]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw)
    X_test  = scaler.transform(X_test_raw)

    X_train_seq, y_train_seq = make_windows_forecast(X_train, y_train_raw, seq_len)
    X_test_seq, y_test_seq   = make_windows_forecast(X_test, y_test_raw, seq_len)

    train_ds = ForecastDataset(X_train_seq, y_train_seq)
    test_ds  = ForecastDataset(X_test_seq, y_test_seq)

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=False)
    test_loader  = DataLoader(test_ds, batch_size=64, shuffle=False)

    model = TransformerForecaster(n_features=len(feature_cols))
    history = train_forecaster(model, train_loader, epochs=epochs, lr=lr)

    # collect test predictions
    model.eval()
    preds_test = []

    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(DEVICE)
            pred = model(xb).detach().cpu().numpy()
            preds_test.extend(pred)

    preds_test = np.array(preds_test)

    return {
        "model": model,
        "scaler": scaler,
        "X_raw": X_raw,
        "y_raw": y_raw,
        "X_test_seq": X_test_seq,
        "y_test_seq": y_test_seq,
        "preds_test": preds_test,
        "history": history,
        "temp": temp,
        "horizon": horizon,
        "seq_len": seq_len
    }

# ==============================================================================
# 6. Evaluation & Forecasting Utilities
# ==============================================================================

def forecast_latest_with_intervals(model_pack, feat, feature_cols, confidence_levels=(0.50, 0.78, 0.92)):
    model = model_pack["model"]
    scaler = model_pack["scaler"]
    temp = model_pack["temp"]
    horizon = model_pack["horizon"]
    seq_len = model_pack["seq_len"]

    # Build latest window from the full available feature set
    X_all = scaler.transform(temp[feature_cols])
    latest_window = X_all[-seq_len:]
    latest_window = torch.tensor(latest_window[np.newaxis, :, :], dtype=torch.float32).to(DEVICE)

    model.eval()
    with torch.no_grad():
        pred_log_ret = model(latest_window).item()

    # Convert predicted log return into predicted future price
    last_close = temp["Close"].iloc[-1]
    pred_price = last_close * np.exp(pred_log_ret)

    # Estimate forecast uncertainty from test residuals
    y_test = model_pack["y_test_seq"]
    preds_test = model_pack["preds_test"]
    residuals = y_test - preds_test
    resid_std = np.std(residuals, ddof=1)

    intervals = {}

    for conf in confidence_levels:
        alpha = 1 - conf
        z = NormalDist().inv_cdf(1 - alpha / 2)

        lower_log_ret = pred_log_ret - z * resid_std
        upper_log_ret = pred_log_ret + z * resid_std

        lower_price = last_close * np.exp(lower_log_ret)
        upper_price = last_close * np.exp(upper_log_ret)

        intervals[conf] = {
            "lower_price": lower_price,
            "upper_price": upper_price
        }

    return {
        "horizon": horizon,
        "last_close": last_close,
        "pred_log_ret": pred_log_ret,
        "pred_price": pred_price,
        "resid_std": resid_std,
        "intervals": intervals
    }

def print_forecast_summary(forecast):
    horizon = forecast["horizon"]
    pred_price = forecast["pred_price"]
    pred_log_ret = forecast["pred_log_ret"]

    print(f"\n===== {horizon}-Day Forecast =====")
    print(f"Predicted cumulative log return: {pred_log_ret:.6f}")
    print(f"Forecasted price: ${pred_price:,.2f}")

    for conf, band in forecast["intervals"].items():
        pct = int(round(conf * 100))
        print(
            f"{pct}% prediction interval: "
            f"${band['lower_price']:,.2f} to ${band['upper_price']:,.2f}"
        )

# ==============================================================================
# 7. Visualization Functions (HTML + PNG)
# ==============================================================================

def export_plotly_html(fig, base_filename):
    """Export Plotly figure as HTML."""
    html_filename = f"{base_filename}.html"
    fig.write_html(html_filename)
    print(f"✓ Saved {html_filename}")
    return html_filename

def save_model_performance_png(lstm_roc, lstm_pr, trans_roc, trans_pr, filename):
    """Save model performance comparison as PNG using matplotlib."""
    fig, ax = plt.subplots(figsize=(10, 7))
    
    x = np.arange(2)
    width = 0.35
    
    roc_vals = [lstm_roc, trans_roc]
    pr_vals = [lstm_pr, trans_pr]
    
    rects1 = ax.bar(x - width/2, roc_vals, width, label='ROC-AUC', color='steelblue')
    rects2 = ax.bar(x + width/2, pr_vals, width, label='PR-AUC', color='coral')
    
    ax.set_ylabel('Score')
    ax.set_title('LSTM vs Transformer Model Performance Comparison')
    ax.set_xticks(x)
    ax.set_xticklabels(['LSTM', 'Transformer'])
    ax.legend()
    ax.set_ylim(0, 1.0)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add value labels
    for rect in rects1 + rects2:
        height = rect.get_height()
        ax.annotate(f'{height:.3f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved {filename}")
    return filename

def save_anomaly_scores_png(results, lstm_scores, trans_scores, filename):
    """Save anomaly scores timeline as PNG using matplotlib."""
    fig, ax = plt.subplots(figsize=(14, 7))
    
    x = np.arange(len(results))
    
    ax.plot(x, lstm_scores, label='LSTM Score', color='steelblue', linewidth=1.5, alpha=0.8)
    ax.plot(x, trans_scores, label='Transformer Score', color='coral', linewidth=1.5, alpha=0.8)
    
    # Mark proxy anomalies
    proxy_mask = results["proxy_anomaly"].values == 1
    if proxy_mask.any():
        ax.scatter(x[proxy_mask], trans_scores[proxy_mask], 
                  color='red', marker='x', s=100, label='Proxy Anomaly', zorder=5)
    
    ax.set_xlabel('Time (days)')
    ax.set_ylabel('Reconstruction Error Score')
    ax.set_title('LSTM vs Transformer Anomaly Scores Over Time')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved {filename}")
    return filename

def save_price_with_anomalies_png(results, ticker, filename):
    """Save price with anomaly markers as PNG using matplotlib."""
    fig, ax = plt.subplots(figsize=(14, 7))
    
    x = np.arange(len(results))
    
    # Plot price
    ax.plot(x, results["Close"].values, label=f'{ticker} Close', color='steelblue', linewidth=1.5)
    
    # Mark transformer anomalies
    transformer_top = results["transformer_score"] >= results["transformer_score"].quantile(0.99)
    if transformer_top.any():
        ax.scatter(x[transformer_top], results.loc[transformer_top, "Close"].values, 
                  color='red', s=100, label='Transformer Anomaly', zorder=5, marker='o')
        
        # Add vertical lines for anomaly dates
        for xi in x[transformer_top]:
            ax.axvline(x=xi, color='red', linestyle='--', alpha=0.3, linewidth=1)
    
    ax.set_xlabel('Time (days)')
    ax.set_ylabel('Close Price ($)')
    ax.set_title(f'{ticker} Price with Transformer-Detected Anomalies')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved {filename}")
    return filename

# ==============================================================================
# 8. Main Execution
# ==============================================================================

if __name__ == "__main__":
    
    # Track all exported files
    all_exported_files = []
    
    # -------------------------------------------------------------------------
    # Step 1: Load Data
    # -------------------------------------------------------------------------
    print("\n--- Loading Data ---")
    df, feat, feature_cols, features = load_and_prepare_data(ticker)
    y_proxy = feat["anomaly_label"].values

    # -------------------------------------------------------------------------
    # Step 2: Prepare Sequences for Autoencoders
    # -------------------------------------------------------------------------
    print("\n--- Preparing Sequences ---")
    X_train_seq, y_train_seq, X_test_seq, y_test_seq, scaler, test_indices = prepare_sequences(
        feat, y_proxy, feature_cols, seq_len=SEQ_LEN, split_idx_ratio=TRAIN_SPLIT
    )

    train_ds = SeqDataset(X_train_seq, y_train_seq)
    test_ds  = SeqDataset(X_test_seq, y_test_seq)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False)
    test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

    # -------------------------------------------------------------------------
    # Step 3: Train Autoencoders (LSTM vs Transformer)
    # -------------------------------------------------------------------------
    print("\n--- Training Autoencoders ---")
    
    # LSTM
    lstm_model = LSTMAutoencoder(n_features=len(feature_cols), hidden_size=64, latent_size=32)
    print("Training LSTM...")
    lstm_hist = train_autoencoder(lstm_model, train_loader, epochs=EPOCHS, lr=LEARNING_RATE)

    # Transformer
    transformer_model = TransformerAutoencoder(
        n_features=len(feature_cols),
        d_model=64,
        nhead=4,
        num_layers=2,
        dim_feedforward=128,
        dropout=0.1
    )
    print("\nTraining Transformer...")
    trans_hist = train_autoencoder(transformer_model, train_loader, epochs=EPOCHS, lr=LEARNING_RATE)

    # -------------------------------------------------------------------------
    # Step 4: Evaluate Anomaly Detection
    # -------------------------------------------------------------------------
    print("\n--- Evaluating Anomaly Detection ---")
    lstm_scores, _, _ = reconstruction_scores(lstm_model, test_loader)
    trans_scores, _, _ = reconstruction_scores(transformer_model, test_loader)

    lstm_roc_auc = roc_auc_score(y_test_seq, lstm_scores)
    trans_roc_auc = roc_auc_score(y_test_seq, trans_scores)
    lstm_pr_auc = average_precision_score(y_test_seq, lstm_scores)
    trans_pr_auc = average_precision_score(y_test_seq, trans_scores)

    print("LSTM ROC-AUC:", lstm_roc_auc)
    print("Transformer ROC-AUC:", trans_roc_auc)

    print("LSTM PR-AUC:", lstm_pr_auc)
    print("Transformer PR-AUC:", trans_pr_auc)
    
    # -------------------------------------------------------------------------
    # Step 5: Visualization (Plotly HTML + Matplotlib PNG)
    # -------------------------------------------------------------------------
    print("\n--- Generating Visualizations ---")
    
    # Prepare results dataframe for plotting alignment
    results = pd.DataFrame(index=test_indices)
    results["lstm_score"] = lstm_scores
    results["transformer_score"] = trans_scores
    results["proxy_anomaly"] = y_test_seq
    results["Close"] = feat.loc[test_indices, "Close"]

    try:
        import plotly.graph_objects as go

        # === Plot 1: Model Performance Comparison ===
        fig = go.Figure()
        fig.add_bar(
            x=["LSTM","Transformer"],
            y=[lstm_roc_auc, trans_roc_auc],
            name="ROC-AUC"
        )
        fig.add_bar(
            x=["LSTM","Transformer"],
            y=[lstm_pr_auc, trans_pr_auc],
            name="PR-AUC"
        )
        fig.update_layout(
            title="Model Performance Comparison",
            template="plotly_white"
        )
        html_file = export_plotly_html(fig, "model_performance_comparison")
        all_exported_files.append(html_file)
        
        # PNG version
        png_file = save_model_performance_png(
            lstm_roc_auc, lstm_pr_auc, trans_roc_auc, trans_pr_auc,
            "model_performance_comparison.png"
        )
        all_exported_files.append(png_file)

        # === Plot 2: Anomaly Scores Timeline ===
        fig_scores = go.Figure()
        fig_scores.add_trace(
            go.Scatter(
                x=results.index,
                y=results["lstm_score"],
                mode="lines",
                name="LSTM score"
            )
        )
        fig_scores.add_trace(
            go.Scatter(
                x=results.index,
                y=results["transformer_score"],
                mode="lines",
                name="Transformer score"
            )
        )
        proxy_mask = results["proxy_anomaly"] == 1
        fig_scores.add_trace(
            go.Scatter(
                x=results.index[proxy_mask],
                y=results.loc[proxy_mask, "transformer_score"],
                mode="markers",
                name="Proxy anomaly day",
                marker=dict(symbol="x", size=10)
            )
        )
        fig_scores.update_layout(
            title="Anomaly Scores on " + ticker,
            xaxis_title="Date",
            yaxis_title="Score",
            template="plotly_white",
            hovermode="x unified",
            width=1100,
            height=500
        )
        html_file = export_plotly_html(fig_scores, "anomaly_scores")
        all_exported_files.append(html_file)
        
        # PNG version
        png_file = save_anomaly_scores_png(
            results, lstm_scores, trans_scores, "anomaly_scores.png"
        )
        all_exported_files.append(png_file)

        # === Plot 3: Price with Anomalies ===
        transformer_top = results["transformer_score"] >= results["transformer_score"].quantile(0.99)
        anomaly_dates = results.index[transformer_top]

        fig_price = go.Figure()
        fig_price.add_trace(
            go.Scatter(
                x=results.index,
                y=results["Close"],
                mode="lines",
                name= ticker + " Close"
            )
        )
        fig_price.add_trace(
            go.Scatter(
                x=results.index[transformer_top],
                y=results.loc[transformer_top, "Close"],
                mode="markers",
                name="Transformer flagged anomaly",
                marker=dict(color="red", size=10)
            )
        )
        for dt in anomaly_dates:
            fig_price.add_vline(
                x=dt,
                line_width=1,
                line_dash="dash",
                line_color="red",
                opacity=0.35
            )
        fig_price.update_layout(
            title= ticker + " Price with Transformer-Detected Anomalies",
            xaxis_title="Date",
            yaxis_title="Close Price",
            template="plotly_white",
            hovermode="x unified",
            width=1150,
            height=550
        )
        html_file = export_plotly_html(fig_price, "price_with_anomalies")
        all_exported_files.append(html_file)
        
        # PNG version
        png_file = save_price_with_anomalies_png(
            results, ticker, "price_with_anomalies.png"
        )
        all_exported_files.append(png_file)

    except ImportError:
        print("Plotly not installed. Skipping HTML visualizations.")
        
        # Still generate PNG versions
        png_file = save_model_performance_png(
            lstm_roc_auc, lstm_pr_auc, trans_roc_auc, trans_pr_auc,
            "model_performance_comparison.png"
        )
        all_exported_files.append(png_file)
        
        png_file = save_anomaly_scores_png(
            results, lstm_scores, trans_scores, "anomaly_scores.png"
        )
        all_exported_files.append(png_file)
        
        png_file = save_price_with_anomalies_png(
            results, ticker, "price_with_anomalies.png"
        )
        all_exported_files.append(png_file)

    # -------------------------------------------------------------------------
    # Step 6: Forecasting (Transformer)
    # -------------------------------------------------------------------------
    print("\n--- Training Forecasters ---")
    print("training epochs for 1-day future/forecast stock price:")
    model_pack_1d = train_horizon_model(feat, feature_cols, horizon=1, seq_len=30, epochs=EPOCHS, lr=LEARNING_RATE)

    print("training epochs for 5-day future/forecast stock price:")
    model_pack_5d = train_horizon_model(feat, feature_cols, horizon=5, seq_len=30, epochs=EPOCHS, lr=LEARNING_RATE)

    print("training epochs for 21-day future/forecast stock price:")
    model_pack_21d = train_horizon_model(feat, feature_cols, horizon=21, seq_len=30, epochs=EPOCHS, lr=LEARNING_RATE)

    print("All three models trained.")

    forecast_1d = forecast_latest_with_intervals(model_pack_1d, feat, feature_cols)
    forecast_5d = forecast_latest_with_intervals(model_pack_5d, feat, feature_cols)
    forecast_21d = forecast_latest_with_intervals(model_pack_21d, feat, feature_cols)

    print_forecast_summary(forecast_1d)
    print_forecast_summary(forecast_5d)
    print_forecast_summary(forecast_21d)

    # -------------------------------------------------------------------------
    # Step 7: Summary
    # -------------------------------------------------------------------------
    print("\n" + "="*80)
    print("Analysis Complete!")
    print("="*80)
    
    html_files = [f for f in all_exported_files if f.endswith('.html')]
    png_files = [f for f in all_exported_files if f.endswith('.png')]
    
    print(f"\n✓ Exported {len(all_exported_files)} files:")
    print(f"  - {len(html_files)} HTML files (interactive)")
    print(f"  - {len(png_files)} PNG files (static)")
    
    if html_files:
        print("\nHTML Files:")
        for f in html_files:
            print(f"  ✓ {f}")
    
    if png_files:
        print("\nPNG Files:")
        for f in png_files:
            print(f"  ✓ {f}")
    
    # -------------------------------------------------------------------------
    # Analysis & Interpretation Comments
    # -------------------------------------------------------------------------
    print("\n" + "="*80)
    print("INTERPRETATION & ANALYSIS")
    print("="*80)
    print("""
    How to interpret whether LSTM performs better or Transformer performs better:

    If the transformer is superior in your run, you will usually see one or more of these:
    - lower average reconstruction error on holdout windows
    - higher ROC-AUC / PR-AUC against the proxy anomaly labels
    - anomaly spikes that line up more cleanly with major volatility events
    - less "smearing" of error across nearby normal days

    That would support the idea that the transformer captured cross-window structure 
    better than the LSTM.

    Why that can happen:
    The transformer can compare every timestep in the 30-day window with every other 
    timestep directly through self-attention, while the LSTM processes the sequence 
    step by step. That direct access to the full context often helps when anomalies 
    depend on patterns like "a sudden break after a steady trend," "volume regime 
    shift after a quiet period," or "range expansion relative to earlier parts of the window."

    Important caveat:
    If your LSTM wins, that does not mean the experiment failed. On financial time 
    series, data is noisy, non-stationary, and relatively small compared with the 
    scale transformers often like. On smaller samples, LSTM can still be very competitive. 

    A fair conclusion is usually:
    - Transformer often has structural advantages
    - LSTM may still perform as well or better on limited data
    - Everything is measured on a case-by-case basis

    If LSTM wins, it suggests:
    - anomalies depend mostly on recent history
    - patterns are short-term
    - dataset may be small

    If Transformer wins, it suggests:
    - anomalies depend on longer time relationships
    - patterns involve cross-window comparisons
    - the dataset benefits from global context

    When transformers outperform LSTMs:
    Transformers perform best when:
    - long sequences
    - multiple signals
    - complex seasonality
    - long-range dependencies

    Example use cases:
    - stock price forecasting
    - energy demand prediction
    - climate modeling
    - IoT sensor analysis
    - anomaly detection in industrial equipment

    Feature Comparison:
    Feature	                  LSTM	         Transformer
    Sequential	                Yes	             No
    Parallelizable	            No	             Yes
    Long-range relationships	  Limited	         Excellent
    Training speed	            Slower	         Faster on GPU
    Data efficiency	            Good	         Requires more data
    Interpretability	          Hard	         Attention maps help

    Use case	                    Model often chosen
    small dataset	                LSTM
    large sensor networks	        Transformer
    financial microstructure	    often LSTM
    macro forecasting	            often Transformer
    """)
    print("================================================================================")
    print("End of Script")
    print("================================================================================")

Using device: cuda

--- Loading Data ---
Data shape: (1256, 5)
                                 Open        High         Low       Close  \
Date                                                                        
2026-03-04 00:00:00-05:00  401.269989  411.029999  400.309998  405.200012   
2026-03-05 00:00:00-05:00  404.420013  411.609985  404.399994  410.679993   
2026-03-06 00:00:00-05:00  409.200012  413.049988  408.510010  408.959991   
2026-03-09 00:00:00-04:00  404.920013  410.209991  403.500000  409.410004   
2026-03-10 00:00:00-04:00  410.029999  410.200012  406.357697  406.739990   

                             Volume  
Date                                 
2026-03-04 00:00:00-05:00  35808000  
2026-03-05 00:00:00-05:00  39001300  
2026-03-06 00:00:00-05:00  31123900  
2026-03-09 00:00:00-04:00  30073800  
2026-03-10 00:00:00-04:00   2048392  

--- Preparing Sequences ---
Train shape: (842, 30, 6), Test shape: (344, 30, 6)

--- Training Autoencoders ---
Training LSTM...
Ep